# ChaoSSM — Phase 3a-ii: Is `mamba2`'s Copy Failure a Hyperparameter Artifact?

## Why this phase

Phase 3a found `mamba2` at 0.0599 accuracy on copy — statistically indistinguishable from
chance (0.0625) and from `linear` (p=0.41), while both chaotic arms beat it decisively
(p=0.0078, n=8, the floor). Taken at face value this is a stronger result than the project
already had: chaos beats genuine selectivity, not just a non-selective linear baseline.

But real Mamba/Mamba-2 is specifically known for strong copying performance — that is one of
the selective mechanism's most celebrated properties in the literature this project's own
introduction cites. A selective-SSM landing at exact chance is surprising enough to check for
a confound before trusting it, especially since this project already hit exactly this trap
once: Phase 2d found the chaotic map's apparent failure at larger scale was an untuned
hyperparameter artifact (fixed by retuning gamma and learning rate), not a real limitation.
Phase 3a used the same `lr=3e-4` for all four arms, untuned for `mamba2` specifically.

One structural reason to suspect a capacity issue rather than a training issue: this
simplified `mamba2` compresses the entire `d_model=48` input into a *single* `state_dim=48`
vector shared across all channels. Real Mamba's copying strength is partly attributed to a
large per-channel state-expansion factor giving many independent memory slots — a single
shared 48-dim state may simply not have room to copy 12 tokens exactly, independent of how
well it's trained.

## Design

A small, cheap sweep — `mamba2` only, copy task only — varying:
- **Learning rate**: {1e-3, 3e-4 (Phase 3a's setting), 1e-4}
- **State dimension** (capacity): {48 (Phase 3a's setting), 96, 192}

5 seeds per configuration (9 combinations x 5 = 45 runs), enough to detect a real effect
without the cost of a full battery rerun. `linear` and `chaossm_free` at their Phase 3a
settings are included once each, as fixed reference lines, not swept.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase3a_ii"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

PHASE3A_REFERENCE = dict(linear=0.0700, chaossm_free=0.1322, chaossm_corridor=0.1576, mamba2_baseline=0.0599)
print("Phase 3a reference (for consistency check):", PHASE3A_REFERENCE)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase3a_ii
Phase 3a reference (for consistency check): {'linear': 0.07, 'chaossm_free': 0.1322, 'chaossm_corridor': 0.1576, 'mamba2_baseline': 0.0599}


## Configuration

In [2]:
SMOKE_TEST = False   # full run: 3 lr x 3 state_dim x 5 seeds for mamba2, + 2 fixed references

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    copy_len: int = 12
    lrs: tuple = (1e-3, 3e-4, 1e-4)
    state_dims: tuple = (48, 96, 192)
    seeds: int = 5

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2
    CFG.lrs = (1e-3, 3e-4); CFG.state_dims = (48, 96)
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "copy_len": "12",
  "lrs": "(0.001, 0.0003, 0.0001)",
  "state_dims": "(48, 96, 192)",
  "seeds": "5"
}


## 1. Model — `mamba2` cell unchanged; `linear`/`chaossm_free` kept only as fixed references

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)


class Mamba2Cell(nn.Module):
    # Selective-SSM baseline, unchanged from Phase 3a. Delta_t and B_t are functions of
    # the current input; A is a fixed, learnable per-channel decay rate.
    def __init__(self, cfg):
        super().__init__()
        self.N = cfg.state_dim
        A_init = torch.arange(1, cfg.state_dim + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A_init))
        self.W_delta = nn.Linear(cfg.d_model, cfg.state_dim)
        self.W_B = nn.Linear(cfg.d_model, cfg.state_dim)
        nn.init.zeros_(self.W_delta.bias); nn.init.zeros_(self.W_B.bias)
    def step(self, h, x):
        delta = F.softplus(self.W_delta(x))
        A = -torch.exp(self.A_log)
        A_bar = torch.exp(delta * A)
        B_t = self.W_B(x)
        return A_bar * h + delta * B_t
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev, dtype=self.A_log.dtype)


def make_cell(cfg, arm):
    if arm == "mamba2":
        return Mamba2Cell(cfg)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    return ChaoSSMCell(cfg, phi_kind)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, arm, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.arm = arm
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = make_cell(cfg, arm)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs = []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        return torch.stack(outs, 1)

## 2. Task and accuracy — unchanged

In [4]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

## 3. Trainer — simplified (no Lyapunov machinery needed for this diagnostic)

In [5]:
def train_one(arm, lr, state_dim, seed, cfg):
    cfg = Config(**{**asdict(cfg), "lr": lr, "state_dim": state_dim, "d_model": state_dim})
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg, arm, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))
    diverged = False; final_acc = float("nan")

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()
        if not (math.isfinite(loss.item()) and math.isfinite(gn)):
            diverged = True; break
        if step % 20 == 0:
            with torch.no_grad():
                final_acc = copy_accuracy(logits, tgt)

    return dict(arm=arm, lr=lr, state_dim=state_dim, seed=seed, diverged=diverged, final_acc=final_acc)

## 4. Run — mamba2 sweep, plus fixed references

In [6]:
rows = []
total = len(CFG.lrs) * len(CFG.state_dims) * CFG.seeds + 2 * CFG.seeds
done = 0
for lr in CFG.lrs:
    for sd_dim in CFG.state_dims:
        for sd in range(CFG.seeds):
            r = train_one("mamba2", lr, sd_dim, sd, CFG)
            rows.append(r); done += 1
            flag = " DIVERGED" if r["diverged"] else ""
            print(f"[{done}/{total}] mamba2  lr={lr:.0e}  N={sd_dim:4d}  seed={sd}  "
                  f"final_acc={r['final_acc']:.4f} (chance {CHANCE:.4f}){flag}")

for arm in ("linear", "chaossm_free"):
    for sd in range(CFG.seeds):
        r = train_one(arm, CFG.lr, CFG.state_dim, sd, CFG)
        rows.append(r); done += 1
        print(f"[{done}/{total}] {arm:14s} (reference)  seed={sd}  final_acc={r['final_acc']:.4f}")

SWEEP = pd.DataFrame(rows)
savetable(SWEEP, "phase3a_ii_sweep.csv")

[1/55] mamba2  lr=1e-03  N=  48  seed=0  final_acc=0.1146 (chance 0.0625)
[2/55] mamba2  lr=1e-03  N=  48  seed=1  final_acc=0.1484 (chance 0.0625)
[3/55] mamba2  lr=1e-03  N=  48  seed=2  final_acc=0.0612 (chance 0.0625)
[4/55] mamba2  lr=1e-03  N=  48  seed=3  final_acc=0.0794 (chance 0.0625)
[5/55] mamba2  lr=1e-03  N=  48  seed=4  final_acc=0.1211 (chance 0.0625)
[6/55] mamba2  lr=1e-03  N=  96  seed=0  final_acc=0.1823 (chance 0.0625)
[7/55] mamba2  lr=1e-03  N=  96  seed=1  final_acc=0.1719 (chance 0.0625)
[8/55] mamba2  lr=1e-03  N=  96  seed=2  final_acc=0.1536 (chance 0.0625)
[9/55] mamba2  lr=1e-03  N=  96  seed=3  final_acc=0.1771 (chance 0.0625)
[10/55] mamba2  lr=1e-03  N=  96  seed=4  final_acc=0.1706 (chance 0.0625)
[11/55] mamba2  lr=1e-03  N= 192  seed=0  final_acc=0.1628 (chance 0.0625)
[12/55] mamba2  lr=1e-03  N= 192  seed=1  final_acc=0.1992 (chance 0.0625)
[13/55] mamba2  lr=1e-03  N= 192  seed=2  final_acc=0.1810 (chance 0.0625)
[14/55] mamba2  lr=1e-03  N= 192  

## 5. Does any configuration let mamba2 learn copying?

In [7]:
mamba_only = SWEEP[SWEEP.arm == "mamba2"]
summary = mamba_only.groupby(["lr","state_dim"]).final_acc.agg(["mean","std"]).round(4)
print(summary.to_string())
savetable(summary.reset_index(), "phase3a_ii_summary.csv")

ref_free = SWEEP[SWEEP.arm=="chaossm_free"].final_acc.mean()
ref_linear = SWEEP[SWEEP.arm=="linear"].final_acc.mean()
best_config = mamba_only.groupby(["lr","state_dim"]).final_acc.mean().idxmax()
best_acc = mamba_only.groupby(["lr","state_dim"]).final_acc.mean().max()
print(f"\nbest mamba2 configuration: lr={best_config[0]:.0e}, N={best_config[1]} -> acc={best_acc:.4f}")
print(f"reference: linear={ref_linear:.4f}  chaossm_free={ref_free:.4f}  chance={CHANCE:.4f}")

any_learned = bool(best_acc > CHANCE * 1.5)
matches_chaos = bool(best_acc > ref_free * 0.8)
print(f"\nany mamba2 config clears 1.5x chance: {any_learned}")
print(f"best mamba2 config reaches >=80% of chaossm_free's accuracy: {matches_chaos}")

                    mean     std
lr     state_dim                
0.0001 48         0.0581  0.0076
       96         0.0628  0.0084
       192        0.0648  0.0101
0.0003 48         0.0581  0.0112
       96         0.0690  0.0142
       192        0.1125  0.0297
0.0010 48         0.1049  0.0347
       96         0.1711  0.0108
       192        0.1826  0.0134
saved /content/drive/My Drive/ChaoSSM/results/phase3a_ii/phase3a_ii_summary.csv

best mamba2 configuration: lr=1e-03, N=192 -> acc=0.1826
reference: linear=0.0638  chaossm_free=0.1367  chance=0.0625

any mamba2 config clears 1.5x chance: True
best mamba2 config reaches >=80% of chaossm_free's accuracy: True


In [8]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for lr in CFG.lrs:
    d = mamba_only[mamba_only.lr == lr].groupby("state_dim").final_acc.mean()
    ax[0].plot(d.index, d.values, marker="o", label=f"lr={lr:.0e}")
ax[0].axhline(CHANCE, color="r", ls="--", label="chance")
ax[0].axhline(ref_free, color="g", ls=":", label="chaossm_free (Phase 3a)")
ax[0].set(xlabel="state_dim", ylabel="copy accuracy", title="mamba2: does capacity fix copying?")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for sd_dim in CFG.state_dims:
    d = mamba_only[mamba_only.state_dim == sd_dim].groupby("lr").final_acc.mean()
    ax[1].plot([f"{lr:.0e}" for lr in d.index], d.values, marker="o", label=f"N={sd_dim}")
ax[1].axhline(CHANCE, color="r", ls="--", label="chance")
ax[1].axhline(ref_free, color="g", ls=":", label="chaossm_free (Phase 3a)")
ax[1].set(xlabel="learning rate", ylabel="copy accuracy", title="mamba2: does learning rate fix copying?")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase3a_ii_diagnostic.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase3a_ii/phase3a_ii_diagnostic.png


## 6. Verdict

| Observation | Reading |
|---|---|
| Some configuration clears ~1.5x chance and approaches chaossm_free's level | Phase 3a's result was a hyperparameter/capacity artifact. Re-run Phase 3a with the corrected mamba2 configuration before drawing any conclusion about chaos vs. selectivity, and use the fixed recipe in Phase 3b. |
| All configurations stay near chance regardless of lr or state_dim | The failure is structural, not tunable within this sweep's range — this simplified single-state-vector adaptation may not capture the inductive bias that gives real multi-head Mamba-2 its copying strength. This is now a defensible, checked finding rather than an untested default, and should be reported with this diagnostic as evidence. |
| Larger state_dim helps but lr doesn't (or vice versa) | Isolates which factor matters, informing what to fix before Phase 3b rather than guessing. |

In [9]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    phase3a_reference=PHASE3A_REFERENCE,
    sweep_summary=summary.reset_index().to_dict("records"),
    best_config=dict(lr=float(best_config[0]), state_dim=int(best_config[1]), acc=float(best_acc)),
    reference_linear=float(ref_linear), reference_chaossm_free=float(ref_free),
    any_learned=any_learned, matches_chaos=matches_chaos,
    decision="Phase 3a's result was a fixable hyperparameter/capacity artifact -- redo with corrected config" if any_learned
             else "mamba2's copy failure is structural within this sweep's range -- report as a checked, defensible finding",
    config=dict(steps=CFG.steps, seeds=CFG.seeds, lrs=CFG.lrs, state_dims=CFG.state_dims),
)
savejson(verdict, "phase3a_ii_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase3a_ii/phase3a_ii_verdict.json
{
  "smoke_test": false,
  "phase3a_reference": {
    "linear": 0.07,
    "chaossm_free": 0.1322,
    "chaossm_corridor": 0.1576,
    "mamba2_baseline": 0.0599
  },
  "sweep_summary": [
    {
      "lr": 0.0001,
      "state_dim": 48,
      "mean": 0.0581,
      "std": 0.0076
    },
    {
      "lr": 0.0001,
      "state_dim": 96,
      "mean": 0.0628,
      "std": 0.0084
    },
    {
      "lr": 0.0001,
      "state_dim": 192,
      "mean": 0.0648,
      "std": 0.0101
    },
    {
      "lr": 0.0003,
      "state_dim": 48,
      "mean": 0.0581,
      "std": 0.0112
    },
    {
      "lr": 0.0003,
      "state_dim": 96,
      "mean": 0.069,
      "std": 0.0142
    },
    {
      "lr": 0.0003,
      "state_dim": 192,
      "mean": 0.1125,
      "std": 0.0297
    },
    {
      "lr": 0.001,
      "state_dim": 48,
      "mean": 0.1049,
      "std": 0.0347
    },
    {
      "lr": 0.001,
      "state_dim": 96,